# Q5 — Ensemble Analysis: Human vs AI Text Detection

**Cornell Tech · CS5785 Kaggle Competition**

This notebook evaluates whether averaging probabilities from three independently developed classifiers improves AUROC on the **same 2,000 labeled validation examples**. It does **not** retrain the original three models, and it does not access hidden Kaggle labels.

- Model 1: Word TF-IDF + Logistic Regression (independently reproduced validation predictions)
- Model 2: Character TF-IDF + sigmoid-calibrated Linear SVM
- Model 3: ConvoKit stylometric features + Logistic Regression

**Experimental protocol:** Compare fixed equal-weight baselines first. Weighted models explored on this validation set are reported only as exploratory findings, not independent evidence of test-set improvement.

**Files needed:** `model1_reproduced_validation_predictions.csv`, `validation_predictions.csv`, and `model3_val_predictions.csv`. These are local validation artifacts; **do not commit the CSVs to the public course GitHub repository**.

## 1. Environment and input files

I load the Python libraries for data alignment, AUROC calculation, and plotting. The input CSVs contain predictions already generated on the held-out validation set, so running this notebook does not require re-extracting ConvoKit features.

In Colab, use the Files sidebar to upload the three CSVs next to this notebook, or upload them when prompted below.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from IPython.display import display

INPUTS = {
    "m1": Path("model1_reproduced_validation_predictions.csv"),
    "m2": Path("validation_predictions.csv"),
    "m3": Path("model3_val_predictions.csv"),
}

missing = [str(p) for p in INPUTS.values() if not p.exists()]
if missing:
    try:
        from google.colab import files
        print("Please upload these files:", missing)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Required CSVs missing: {missing}")
assert all(path.exists() for path in INPUTS.values()), "One or more prediction CSVs are missing."
print("All input files are available.")

## 2. Align validation predictions and verify labels

I merge the three models' predictions using **the original text ID**, rather than relying on CSV row order. I also verify that all three models use exactly the same 2,000 examples and matching true labels. This prevents an invalid comparison caused by inconsistent validation splits or row order.

In [ ]:
m1 = pd.read_csv(INPUTS["m1"], dtype={"id": str}).rename(columns={"m1_prob": "model1_p_ai"})
m2 = pd.read_csv(INPUTS["m2"], dtype={"id": str}).rename(columns={"probability": "model2_p_ai"})
m3 = pd.read_csv(INPUTS["m3"], dtype={"id": str}).rename(columns={"p_ai": "model3_p_ai"})

for name, df, col in [
    ("Model 1", m1, "model1_p_ai"),
    ("Model 2", m2, "model2_p_ai"),
    ("Model 3", m3, "model3_p_ai"),
]:
    assert {"id", "label", col}.issubset(df.columns), f"Missing columns in {name}"
    assert len(df) == 2000 and df["id"].is_unique, f"Unexpected validation IDs in {name}"
    assert df[col].notna().all() and df[col].between(0, 1).all(), f"Invalid probabilities in {name}"

assert set(m1.id) == set(m2.id) == set(m3.id), "The models use different validation IDs."

combined = (
    m1[["id", "label", "model1_p_ai"]]
    .merge(m2[["id", "label", "model2_p_ai"]], on=["id", "label"], validate="one_to_one")
    .merge(m3[["id", "label", "model3_p_ai"]], on=["id", "label"], validate="one_to_one")
    .sort_values("id", key=lambda ids: ids.astype(int))
    .reset_index(drop=True)
)
assert len(combined) == 2000, "Labels disagree or some IDs were dropped during merging."
assert combined.label.value_counts().to_dict() == {0: 1000, 1: 1000}
print("Verified: 2,000 matching IDs; 1,000 human + 1,000 AI; labels agree.")
display(combined.head())

## 3. Compare each base model with simple ensembles

I calculate AUROC for each standalone model and for fixed probability averages. In the primary comparison, ensemble weights are **fixed in advance at equal values**; I do not use Kaggle test labels to choose them.

I also include two explicitly **exploratory** weighted combinations. Their weights were examined on the same validation set, so a small improvement in this table should not be treated as confirmed out-of-sample progress.

In [ ]:
y = combined["label"].to_numpy()
p1 = combined["model1_p_ai"].to_numpy()
p2 = combined["model2_p_ai"].to_numpy()
p3 = combined["model3_p_ai"].to_numpy()

configurations = {
    "Model 1 — Word TF-IDF + LR": p1,
    "Model 2 — Character TF-IDF + SVM": p2,
    "Model 3 — ConvoKit + LR": p3,
    "Model 1 + 2 (50/50)": (p1 + p2) / 2,
    "Model 2 + 3 (50/50)": (p2 + p3) / 2,
    "Model 1 + 2 + 3 (equal weights)": (p1 + p2 + p3) / 3,
    "Exploratory: M2 75% + M3 25%": 0.75 * p2 + 0.25 * p3,
    "Exploratory: M1 20% + M2 60% + M3 20%": 0.20 * p1 + 0.60 * p2 + 0.20 * p3,
}

baseline_auc = roc_auc_score(y, p2)
summary = pd.DataFrame([
    {"Configuration": name,
     "Validation AUROC": roc_auc_score(y, prob),
     "Difference vs Model 2": roc_auc_score(y, prob) - baseline_auc,
     "Exploratory weight choice": name.startswith("Exploratory")}
    for name, prob in configurations.items()
])
display(summary.round(6))

## 4. Measure probability-aggregation overhead

I time only the NumPy operation that combines cached probability arrays. This is not model training time or AUROC evaluation time. The values are hardware-dependent and should not be compared as if they measure end-to-end inference.


In [ ]:
def median_aggregation_ms(weights, repetitions=1000):
    durations = []
    for _ in range(repetitions):
        t0 = time.perf_counter_ns()
        _ = weights[0]*p1 + weights[1]*p2 + weights[2]*p3
        durations.append((time.perf_counter_ns() - t0) / 1_000_000)
    return float(np.median(durations))

weights_by_name = {
    "Model 1 — Word TF-IDF + LR": (1, 0, 0),
    "Model 2 — Character TF-IDF + SVM": (0, 1, 0),
    "Model 3 — ConvoKit + LR": (0, 0, 1),
    "Model 1 + 2 (50/50)": (.5, .5, 0),
    "Model 2 + 3 (50/50)": (0, .5, .5),
    "Model 1 + 2 + 3 (equal weights)": (1/3, 1/3, 1/3),
    "Exploratory: M2 75% + M3 25%": (0, .75, .25),
    "Exploratory: M1 20% + M2 60% + M3 20%": (.2, .6, .2),
}
runtimes = pd.DataFrame([
    {"Configuration": name, "Aggregation overhead (ms, median)": median_aggregation_ms(weights)}
    for name, weights in weights_by_name.items()
])
display(runtimes.round(5))
print("Only aggregation of cached validation probabilities is timed; no training/inference/AUROC timing.")


## 5. Plot the observed validation results

I plot the standalone and fixed equal-weight ensemble AUROCs for a clear visual comparison. The x-axis labels distinguish the different model combinations. This figure describes performance on one validation split only.

In [ ]:
fixed = summary.loc[~summary["Exploratory weight choice"]].copy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fixed["Configuration"], fixed["Validation AUROC"])
ax.invert_yaxis()
ax.set_xlim(0.8, 1.0)
ax.set_xlabel("Validation AUROC (axis begins at 0.80)")
ax.set_title("Model Comparison and Equal-weight Ensembles")
for i, auc in enumerate(fixed["Validation AUROC"]):
    ax.text(min(auc + 0.002, 0.99), i, f"{auc:.4f}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig("q5_ensemble_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Optional paired bootstrap sensitivity check

I resample positive and negative validation examples separately, preserving class counts. I compare the exploratory weighted three-model ensemble against Model 2's predictions using the **same resampled IDs** in each replicate.

**Important limitation:** The weights were selected after examining this validation set. The bootstrap interval below is therefore only descriptive; it does not adjust for the selection of ensemble weights and should not be presented as an independently validated improvement.

In [ ]:
rng = np.random.default_rng(42)
positives = np.flatnonzero(y == 1)
negatives = np.flatnonzero(y == 0)
weighted = configurations["Exploratory: M1 20% + M2 60% + M3 20%"]

bootstrap_differences = []
for _ in range(1000):
    sample_index = np.concatenate([
        rng.choice(negatives, len(negatives), replace=True),
        rng.choice(positives, len(positives), replace=True)
    ])
    delta = (roc_auc_score(y[sample_index], weighted[sample_index])
             - roc_auc_score(y[sample_index], p2[sample_index]))
    bootstrap_differences.append(delta)

ci_low, ci_high = np.percentile(bootstrap_differences, [2.5, 97.5])
observed_delta = roc_auc_score(y, weighted) - baseline_auc
print(f"Observed difference: {observed_delta:+.6f}")
print(f"Exploratory bootstrap interval: [{ci_low:+.6f}, {ci_high:+.6f}]")
print("Caution: intervals do not adjust for choosing weights on this same validation set.")

## 7. Export report-ready results and local prediction file

I save a table of validation AUROC results and the comparison plot. The aligned per-text predictions are useful locally for verification and error analysis, but they should **not** be added to the public course GitHub repository because they contain row-level competition labels.

In [ ]:
summary.to_csv("q5_ensemble_summary.csv", index=False)
runtimes.to_csv("q5_evaluation_overhead.csv", index=False)

local_predictions = combined.copy()
for name, probabilities in configurations.items():
    local_predictions[name] = probabilities
local_predictions.to_csv("q5_aligned_validation_LOCAL_ONLY.csv", index=False)

print("Created q5_ensemble_summary.csv")
print("Created q5_evaluation_overhead.csv")
print("Created q5_ensemble_comparison.png")
print("Created q5_aligned_validation_LOCAL_ONLY.csv — do not publish to GitHub")

## 8. Writing notes for Q5.1–Q5.4

- **Q5.1 — Hypothesis:** Do stylistic and word-level predictions contribute complementary ranking information beyond character n-grams?
- **Q5.2 — Method:** Fixed 8,000/2,000 labeled split; three already fitted models; ID-aligned probabilities; no test labels; 50/50 and three-way equal-weight comparisons first. Additional weights are exploratory.
- **Q5.3 — Results:** Use the AUROC table above and report cached-prediction evaluation overhead separately from model training time.
- **Q5.4 — Interpretation:** Equal-weight ensembles did not improve upon Model 2. Exploratory weighted averaging showed a small validation improvement; because weights were inspected on the same validation partition, its transfer to unseen data remains uncertain. A nested held-out procedure would be needed to confirm it.